
# Finlora Finance Flow – Transaction Fraud Risk Scoring Project

## Project Overview

Finlora Finance Flow is a fintech platform processing transactions across individual and business accounts, multiple payment channels, and different customer segments. As transaction volumes increase, traditional fixed-threshold fraud rules can generate large numbers of false alerts while potentially missing more subtle fraudulent activity.

This project explores the use of machine learning to improve transaction monitoring by assigning each transaction a probability of fraud rather than relying solely on binary rule-based alerts.

### Project Aim

The aim is to develop an explainable fraud risk-scoring model that uses transaction and account-level behavioural information to identify potentially fraudulent transactions and prioritise them for analyst review.

### Key Objectives

 Clean and prepare Finlora's transaction and account data.
 Explore behavioural differences between fraudulent and legitimate transactions.
 Engineer meaningful risk indicators such as transaction amount relative to account baseline, transaction       
 velocity, new-device activity and cross-border activity.
 Develop Logistic Regression as an interpretable baseline model.
 Develop Random Forest as the primary prototype model.
 Compare model performance using Precision, Recall, F1-score and ROC-AUC.
 Analyse different operating thresholds at 70%, 80% and 90% recall.
 Identify the key behavioural features associated with fraud risk.
 Deploy the prototype through a Streamlit dashboard providing a prioritised and explainable transaction-review queue.

### Expected Business Value

The proposed solution is designed to help Finlora:

 Prioritise high-risk transactions for analyst investigation.
 Reduce unnecessary manual review of lower-risk alerts.
 Identify behavioural patterns associated with fraudulent activity.
 Provide analysts with explainable risk indicators rather than black-box predictions.
 Support a more scalable approach to transaction monitoring as transaction volumes grow.

### Project Scope

This is a prototype based on historical transaction and account data. It does not include live payment-system integration, automated transaction blocking, production deployment, automated model retraining or regulatory approval.

The overall workflow is:

**Data Preparation → Exploratory Analysis → Feature Engineering → Model Development → Model Evaluation → Explainability → Streamlit Deployment**

The project evaluates whether machine-learning-based risk scoring can provide Finlora with a more informative and prioritised approach to transaction fraud monitoring.

## Data Cleaning and Preparation

This notebook prepares the Finlora transaction and account datasets for exploratory analysis and subsequent fraud-risk modelling.

**Workflow:** Load data → Standardise structure → Profile data quality → Clean data → Validate → Save cleaned datasets

## Import libraries

In [56]:
import pandas as pd
import numpy as np
from pathlib import Path

##  Define data paths and load the raw datasets

In [57]:
data_folder = Path(r"C:\Users\telvi\Downloads\AMDARI\finlora_finance_flow\data\raw_data")

transactions = pd.read_csv(data_folder / "finlora_transactions.csv")
accounts = pd.read_csv(data_folder / "finlora_accounts.csv")

In [58]:
print("TRANSACTIONS DATASET")
print(f"Rows: {transactions.shape[0]:,}")
print(f"Columns: {transactions.shape[1]:,}")

print("\nACCOUNTS DATASET")
print(f"Rows: {accounts.shape[0]:,}")
print(f"Columns: {accounts.shape[1]:,}")

TRANSACTIONS DATASET
Rows: 126,000
Columns: 24

ACCOUNTS DATASET
Rows: 7,200
Columns: 8


## Standardise column names

Column names are cleaned before further inspection so that later operations use a consistent naming convention.

In [59]:
# Standardise the column names in both datasets
# This makes the column names consistent and easier to use in Python.
for df in [transactions, accounts]:

    # Remove leading and trailing spaces from column names,
    # convert all column names to lowercase,
    # and replace spaces with underscores.
    df.columns = (
        df.columns
        .str.strip()                         # Remove extra spaces
        .str.lower()                         # Convert names to lowercase
        .str.replace(" ", "_", regex=False)  # Replace spaces with underscores
    )

In [60]:
# Display the cleaned column names for the transactions dataset
print("Transaction columns:")
print(transactions.columns.tolist())

Transaction columns:
['transaction_id', 'account_id', 'account_type', 'kyc_tier', 'timestamp', 'day_of_week', 'hour_of_day', 'description', 'merchant_name', 'merchant_category', 'channel', 'amount', 'currency', 'amount_to_avg_ratio', 'avg_transaction_amount_30d', 'transaction_velocity_1h', 'transaction_country', 'home_country', 'is_cross_border', 'device_id', 'is_new_device', 'account_age_days', 'status', 'is_fraud']


In [61]:
# Display the cleaned column names for the accounts dataset
print("\nAccount columns:")
print(accounts.columns.tolist())


Account columns:
['account_id', 'account_holder_name', 'account_type', 'home_country', 'currency', 'kyc_tier', 'account_created_date', 'personal_spend_baseline_usd']


##  Initial data profiling

Check structure, data types and a small sample before cleaning.

In [62]:
# Display a concise summary of the transactions dataset
print("TRANSACTIONS INFO")
transactions.info()

TRANSACTIONS INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 24 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   transaction_id              126000 non-null  object 
 1   account_id                  126000 non-null  object 
 2   account_type                126000 non-null  object 
 3   kyc_tier                    126000 non-null  object 
 4   timestamp                   126000 non-null  object 
 5   day_of_week                 126000 non-null  object 
 6   hour_of_day                 126000 non-null  int64  
 7   description                 126000 non-null  object 
 8   merchant_name               97390 non-null   object 
 9   merchant_category           126000 non-null  object 
 10  channel                     126000 non-null  object 
 11  amount                      126000 non-null  float64
 12  currency                    126000 non-null  object 
 

In [63]:

# Display a concise summary of the accounts dataset
print("\nACCOUNTS INFO")
accounts.info()


ACCOUNTS INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7200 entries, 0 to 7199
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   account_id                   7200 non-null   object 
 1   account_holder_name          7200 non-null   object 
 2   account_type                 7200 non-null   object 
 3   home_country                 7200 non-null   object 
 4   currency                     7200 non-null   object 
 5   kyc_tier                     7200 non-null   object 
 6   account_created_date         7200 non-null   object 
 7   personal_spend_baseline_usd  7200 non-null   float64
dtypes: float64(1), object(7)
memory usage: 450.1+ KB


In [64]:
# Display the first five rows of the transactions dataset
display(transactions.head())

,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,...,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,home_country,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud
0,FLR250216186265,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-02-16 14:55:35,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,...,9143.33,0,NG,NG,0,NaN,NaN,903,Completed,0
1,FLR250423143305,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-23 13:35:40,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,...,9143.33,0,NG,NG,0,NaN,NaN,969,Declined,0
2,FLR250424154897,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-24 12:30:34,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,...,9143.33,0,NG,NG,0,DEV-9055235108,0.0,970,Completed,0
3,FLR250428101772,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-28 14:41:59,Monday,14,CNP PURCHASE - ADOBE CREATIVE CLOUD,Adobe Creative Cloud,Subscription/SaaS,...,14891.16,0,NG,NG,0,NaN,NaN,974,Completed,0
4,FLR250628102827,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-06-28 16:15:47,Saturday,16,MOBILE PURCHASE - APPLE STORE,Apple Store,Electronics,...,49609.59,0,NG,NG,0,DEV-9055235108,0.0,1035,Declined,0


In [65]:
# Display the first five rows of the accounts dataset
display(accounts.head())

,account_id,account_holder_name,account_type,home_country,currency,kyc_tier,account_created_date,personal_spend_baseline_usd
0,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23
1,FLR-ACC-100001,Bluewave Foods Ltd,Business,NG,NGN,Tier1_Basic,2026-04-27,810.68
2,FLR-ACC-100002,Global Trading LLC,Business,US,USD,Tier2_Verified,2026-03-29,433.67
3,FLR-ACC-100003,Femi Adeyemi,Individual,NG,NGN,Tier3_Enhanced,2025-09-15,19.18
4,FLR-ACC-100004,Ngozi Brown,Individual,NG,NGN,Tier3_Enhanced,2024-06-18,28.68


## Data-quality assessment

### Missing values and duplicates

In [66]:
def missing_summary(df):  # Define a function to summarise missing values

    summary = pd.DataFrame({  # Create a DataFrame containing missing-value statistics

        "missing_count": df.isna().sum(),  # Count the missing values in each column

        "missing_pct": (df.isna().mean() * 100).round(2)  # Calculate missing values as a percentage
    })

    return summary.sort_values("missing_pct", ascending=False)  # Sort columns from highest to lowest missing percentage

In [67]:


print("Transaction missing values:")  # Display a heading for the transaction missing-value summary

display(missing_summary(transactions))  # Display the number and percentage of missing values in the transactions dataset


Transaction missing values:


,missing_count,missing_pct
merchant_name,28610,22.71
is_new_device,14334,11.38
device_id,14334,11.38
transaction_id,0,0.00
amount_to_avg_ratio,0,0.00
status,0,0.00
account_age_days,0,0.00
is_cross_border,0,0.00
home_country,0,0.00
transaction_country,0,0.00


In [68]:
print("Account missing values:")  # Display a heading for the account missing-value summary

display(missing_summary(accounts))  # Display the number and percentage of missing values in the accounts dataset

Account missing values:


,missing_count,missing_pct
account_id,0,0.0
account_holder_name,0,0.0
account_type,0,0.0
home_country,0,0.0
currency,0,0.0
kyc_tier,0,0.0
account_created_date,0,0.0
personal_spend_baseline_usd,0,0.0


In [69]:
# Check both datasets for completely duplicated rows to identify
# potential duplicate records that could affect data quality and
# lead to inaccurate analysis or model results.

transaction_duplicates = transactions.duplicated().sum()

print(f"Transaction duplicate rows: {transaction_duplicates:,}")


Transaction duplicate rows: 0


In [70]:
account_duplicates = accounts.duplicated().sum()

print(f"Account duplicate rows: {account_duplicates:,}")

Account duplicate rows: 0


##  Convert date and time fields

Convert timestamp and account creation dates to proper datetime types. Invalid dates are coerced to missing values and then checked.

In [71]:
transactions["timestamp"] = pd.to_datetime(
    transactions["timestamp"],
    errors="coerce"
)

accounts["account_created_date"] = pd.to_datetime(
    accounts["account_created_date"],
    errors="coerce"
)

print("Invalid transaction timestamps:", transactions["timestamp"].isna().sum())
print("Invalid account creation dates:", accounts["account_created_date"].isna().sum())

Invalid transaction timestamps: 0
Invalid account creation dates: 0


## Clean text fields

Remove leading and trailing whitespace from text columns. IDs and free-text values are not lowercased because their original form may be meaningful.

In [72]:
# Remove leading and trailing whitespace from all text-based columns
# in both datasets to ensure consistent and clean categorical values.

for df in [transactions, accounts]:

    text_columns = df.select_dtypes(include="object").columns

    for col in text_columns:

        df[col] = df[col].str.strip()

print("Leading and trailing spaces removed from text fields.")

Leading and trailing spaces removed from text fields.


##  Standardise genuine categorical variables

The following transaction fields are genuine categorical variables. Their text is standardised to lowercase to prevent duplicate categories caused by capitalisation differences.

In [73]:
categorical_columns = [  # Define the categorical columns that need to be standardised and checked
    "channel",
    "account_type",
    "kyc_tier",
    "merchant_category"
]

for col in categorical_columns:  # Loop through each categorical column

    if col in transactions.columns:  # Check that the column exists in the transactions dataset

        transactions[col] = transactions[col].str.lower()  # Convert all text values to lowercase for consistency

for col in categorical_columns:  # Loop through the categorical columns again to inspect their values

    if col in transactions.columns:  # Check that the column exists before analysing it

        print(f"\n--- {col} ---")  # Print the column name as a heading

        display(transactions[col].value_counts(dropna=False))  # Display the frequency of each category, including missing values


--- channel ---


channel
mobile app          41615
web dashboard       31330
card not present    17422
card present        15080
api/integration     13083
ussd                 7470
Name: count, dtype: int64


--- account_type ---


account_type
individual    82112
business      43888
Name: count, dtype: int64


--- kyc_tier ---


kyc_tier
tier2_verified    64094
tier3_enhanced    39094
tier1_basic       22812
Name: count, dtype: int64


--- merchant_category ---


merchant_category
payroll transfer     17117
retail               13742
groceries            13650
utilities            12444
subscription/saas    12006
p2p transfer         10715
restaurants           8416
wire transfer         7639
travel                7416
electronics           6173
atm withdrawal        5129
insurance             4528
healthcare            4382
crypto exchange       1692
gambling/gaming        951
Name: count, dtype: int64

##  Handle missing merchant and device information

Missing merchant and device identifiers are retained rather than deleting the associated transactions. `Unknown` indicates that the information was not available.

In [74]:
# Explore the relationship between merchant names and merchant categories
# to identify which merchants belong to each transaction category.

merchant_category_summary = (
    transactions[["merchant_name", "merchant_category"]]
    .value_counts(dropna=False)
    .reset_index(name="transaction_count")
)

display(merchant_category_summary)

,merchant_name,merchant_category,transaction_count
0,NaN,payroll transfer,17117
1,NaN,p2p transfer,10715
2,International Wire,wire transfer,7584
3,Jumia,retail,2365
4,Apple Store Online,retail,2311
...,...,...,...
73,NaN,atm withdrawal,42
74,NaN,insurance,39
75,NaN,healthcare,36
76,NaN,crypto exchange,14


In [75]:
# Identify transaction categories where a conventional merchant name may not be applicable.

non_merchant_categories = [
    "p2p transfer",
    "atm withdrawal",
    "payroll transfer"
]

# Replace missing merchant names with "Not_Applicable" for these categories
transactions.loc[
    transactions["merchant_category"].isin(non_merchant_categories)
    & transactions["merchant_name"].isna(),
    "merchant_name"
] = "Not_Applicable"

# Treat remaining missing merchant names as genuinely unknown
transactions["merchant_name"] = transactions["merchant_name"].fillna("Unknown")

### Handle `is_new_device`

The source data contain the same number of missing `device_id` and `is_new_device` values. Check this relationship before assigning an explicit unknown category.

In [76]:
device_relationship = pd.crosstab(
    transactions["device_id"].eq("Unknown"),
    transactions["is_new_device"].isna()
)

display(device_relationship)

is_new_device,False,True
device_id,,
False,111666,14334


In [77]:
# 0 = existing device, 1 = new device, -1 = unknown
transactions["is_new_device"] = transactions["is_new_device"].fillna(-1)

print("is_new_device distribution:")
display(transactions["is_new_device"].value_counts().sort_index())

is_new_device distribution:


is_new_device
-1.0     14334
 0.0    107771
 1.0      3895
Name: count, dtype: int64

##  Validate key fields and possible invalid values

These checks identify potential data-quality issues without automatically changing values whose business meaning has not been established.

In [78]:
print("Unique fraud labels:")  # Display a heading for the fraud-label distribution

display(
    transactions["is_fraud"]
    .value_counts(dropna=False)  # Count each fraud label, including any missing values
    .sort_index()                # Sort the fraud labels in ascending order
)


Unique fraud labels:


is_fraud
0    122648
1      3352
Name: count, dtype: int64

In [79]:
# Check the different transaction statuses in the dataset,
# including any missing values, to identify possible data-quality issues.

print("\nUnique transaction statuses:")  # Display a heading for the transaction status distribution

display(transactions["status"].value_counts(dropna=False))  # Count each status, including missing values


Unique transaction statuses:


status
Completed    116703
Declined       5629
Reversed       3668
Name: count, dtype: int64

In [80]:
# Check for negative values in key numerical columns to identify
# potential data-quality issues or invalid values that may need investigation.

print("\nNegative transaction amounts:", (transactions["amount"] < 0).sum())  # Count transactions with negative amounts

print("Negative 30-day average transaction amounts:", (transactions["avg_transaction_amount_30d"] < 0).sum())  # Count negative 30-day average amounts

print("Negative account ages:", (transactions["account_age_days"] < 0).sum())  # Count accounts with negative ages


Negative transaction amounts: 291
Negative 30-day average transaction amounts: 144
Negative account ages: 0


In [81]:
# Check the transaction statuses associated with negative transaction amounts
# to determine whether the negative values may represent legitimate transactions.

negative_amounts = transactions[transactions["amount"] < 0]

print("Negative transaction amounts by status:")

display(
    negative_amounts["status"]
    .value_counts(dropna=False)
)

Negative transaction amounts by status:


status
Completed    274
Declined      12
Reversed       5
Name: count, dtype: int64

In [82]:
# Display key details of negative transactions to investigate
# whether the negative amounts are associated with particular
# transaction types or categories.

display(
    negative_amounts[
        [
            "transaction_id",
            "amount",
            "status",
            "merchant_category",
            "channel",
            "is_cross_border",
            "is_fraud"
        ]
    ]
)

,transaction_id,amount,status,merchant_category,channel,is_cross_border,is_fraud
370,FLR250926151912,-7.13,Completed,retail,web dashboard,0,0
515,FLR260619122573,-419887.90,Completed,retail,mobile app,0,0
762,FLR260510106884,-62.62,Completed,retail,mobile app,0,0
1246,FLR250425175085,-64474.27,Reversed,retail,mobile app,0,0
1288,FLR250811111456,-26708.47,Completed,retail,mobile app,0,0
...,...,...,...,...,...,...,...
123764,FLR250725212601,-40519.69,Declined,retail,mobile app,0,0
124402,FLR251116111628,-67433.26,Completed,retail,card present,0,0
125314,FLR250421185698,-296362.05,Completed,retail,mobile app,0,0
125684,FLR251125180268,-5.12,Completed,retail,mobile app,0,0


In [83]:
# Check the transaction statuses associated with negative 30-day
# average transaction amounts.

negative_avg = transactions[
    transactions["avg_transaction_amount_30d"] < 0
]

print("Negative 30-day average amounts by status:")

display(
    negative_avg["status"]
    .value_counts(dropna=False)
)

Negative 30-day average amounts by status:


status
Completed    135
Declined       9
Name: count, dtype: int64

In [84]:
# Compare negative 30-day average transaction amounts with
# the actual transaction amounts to investigate whether the
# negative averages are consistent with the underlying transactions.

display(
    negative_avg[
        [
            "transaction_id",
            "amount",
            "avg_transaction_amount_30d",
            "amount_to_avg_ratio",
            "status",
            "merchant_category",
            "is_fraud"
        ]
    ]
)

,transaction_id,amount,avg_transaction_amount_30d,amount_to_avg_ratio,status,merchant_category,is_fraud
371,FLR251024157650,6.41,-7.13,0.90,Completed,restaurants,0
763,FLR260518185913,464.42,-25.60,18.14,Completed,payroll transfer,0
1247,FLR250525162454,742611.97,-64474.27,11.52,Completed,electronics,0
1289,FLR250829191546,15787.31,-1674.72,9.43,Completed,retail,0
3278,FLR260226193160,157226.86,-97753.42,1.61,Completed,groceries,0
...,...,...,...,...,...,...,...
123765,FLR250727187117,10298.00,-40519.69,0.25,Completed,restaurants,0
123766,FLR250805223746,24792.00,-15110.84,1.64,Completed,atm withdrawal,0
123767,FLR250807207247,121653.68,-1809.90,67.22,Completed,insurance,0
125854,FLR260428138570,37.00,-14.94,2.48,Completed,p2p transfer,0


In [85]:
# Examine the range and distribution of the negative 30-day average values
# to determine how large the potential data-quality issue is.

display(
    negative_avg["avg_transaction_amount_30d"].describe()
)

count    1.440000e+02
mean    -5.623974e+04
std      1.820492e+05
min     -1.669153e+06
25%     -4.251224e+04
50%     -4.850280e+03
75%     -3.208750e+01
max     -5.900000e-01
Name: avg_transaction_amount_30d, dtype: float64

In [86]:
# Calculate the difference between the actual transaction amount
# and the negative 30-day average to identify unusual discrepancies.

negative_avg = negative_avg.copy()

negative_avg["amount_minus_avg"] = (
    negative_avg["amount"] - negative_avg["avg_transaction_amount_30d"]
)

display(
    negative_avg[
        [
            "transaction_id",
            "amount",
            "avg_transaction_amount_30d",
            "amount_to_avg_ratio",
            "amount_minus_avg",
            "status",
            "merchant_category",
            "is_fraud"
        ]
    ].head(20)
)

,transaction_id,amount,avg_transaction_amount_30d,amount_to_avg_ratio,amount_minus_avg,status,merchant_category,is_fraud
371,FLR251024157650,6.41,-7.13,0.90,13.54,Completed,restaurants,0
763,FLR260518185913,464.42,-25.60,18.14,490.02,Completed,payroll transfer,0
1247,FLR250525162454,742611.97,-64474.27,11.52,807086.24,Completed,electronics,0
1289,FLR250829191546,15787.31,-1674.72,9.43,17462.03,Completed,retail,0
3278,FLR260226193160,157226.86,-97753.42,1.61,254980.28,Completed,groceries,0
3279,FLR260531169301,2148742.80,-97753.42,21.98,2246496.22,Completed,payroll transfer,0
3280,FLR260731157336,-97753.42,-97753.42,1.00,0.00,Completed,retail,0
3281,FLR260820156429,91398.13,-97753.42,0.93,189151.55,Completed,restaurants,0
4131,FLR250227188863,772325.00,-163006.50,4.74,935331.50,Completed,p2p transfer,0
4132,FLR250406126199,5311245.00,-163006.50,32.58,5474251.50,Declined,payroll transfer,0


In [87]:
# Identify accounts associated with negative 30-day average transaction amounts
# to determine whether the issue is concentrated among specific accounts.

negative_avg_accounts = (
    negative_avg
    .groupby("account_id")
    .agg(
        negative_avg_count=("avg_transaction_amount_30d", "size"),
        min_negative_avg=("avg_transaction_amount_30d", "min"),
        max_negative_avg=("avg_transaction_amount_30d", "max")
    )
    .sort_values("negative_avg_count", ascending=False)
)

display(negative_avg_accounts.head(20))

,negative_avg_count,min_negative_avg,max_negative_avg
account_id,,,
FLR-ACC-107067,5,-40519.69,-1809.90
FLR-ACC-100214,5,-163006.50,-163006.50
FLR-ACC-103829,5,-69948.94,-69948.94
FLR-ACC-101372,4,-14959.25,-14959.25
FLR-ACC-100156,4,-97753.42,-97753.42
FLR-ACC-100421,4,-284732.75,-124309.90
FLR-ACC-101500,4,-59.36,-5.45
FLR-ACC-103032,3,-30.78,-12.84
FLR-ACC-103388,3,-34.24,-1.19


In [88]:
# Check how frequently each negative 30-day average value occurs.
# Repeated values may indicate that the feature was calculated at account
# level or that a data-generation process produced the same value repeatedly.

display(
    negative_avg["avg_transaction_amount_30d"]
    .value_counts()
    .head(20)
)

avg_transaction_amount_30d
-69948.94     5
-163006.50    5
-97753.42     4
-14959.25     4
-32.47        3
-10.54        3
-48489.87     3
-25285.26     3
-34547.65     3
-41.49        3
-4850.28      3
-54.47        3
-30173.35     3
-40519.69     3
-284732.75    3
-26.55        3
-280.41       2
-34.24        2
-30.78        2
-124144.68    2
Name: count, dtype: int64

In [89]:
# Investigate the complete transaction history of an account
# associated with negative 30-day average transaction amounts.

account_check = transactions[
    transactions["account_id"] == "FLR-ACC-100214"
]

display(
    account_check[
        [
            "transaction_id",
            "timestamp",
            "amount",
            "avg_transaction_amount_30d",
            "amount_to_avg_ratio",
            "status",
            "merchant_category",
            "is_fraud"
        ]
    ].sort_values("timestamp")
)

,transaction_id,timestamp,amount,avg_transaction_amount_30d,amount_to_avg_ratio,status,merchant_category,is_fraud
4131,FLR250227188863,2025-02-27 14:50:59,772325.00,-163006.50,4.74,Completed,p2p transfer,0
4132,FLR250406126199,2025-04-06 15:45:23,5311245.00,-163006.50,32.58,Declined,payroll transfer,0
4133,FLR250824103787,2025-08-24 14:00:47,15397962.62,-163006.50,94.46,Completed,payroll transfer,1
4134,FLR250924141428,2025-09-24 22:54:16,-163006.50,-163006.50,1.00,Completed,retail,0
4135,FLR251022162041,2025-10-22 09:57:55,7335499.00,-163006.50,45.00,Completed,payroll transfer,0
4136,FLR251209184177,2025-12-09 07:21:06,42607530.34,5113076.00,8.33,Completed,wire transfer,0
4137,FLR260322200710,2026-03-22 17:34:43,2065065.90,5113076.00,0.40,Completed,travel,0
4138,FLR260623175283,2026-06-23 08:26:24,5113076.00,5113076.00,1.00,Completed,travel,0
4139,FLR260623129250,2026-06-23 10:07:51,750619.59,5113076.00,0.15,Completed,retail,0
4140,FLR260816215868,2026-08-16 08:51:28,174957.34,2475034.75,0.07,Completed,subscription/saas,0


In [90]:
# Calculate the actual 30-day rolling average of transaction amounts
# for the selected account and compare it with the existing feature.

account_check = account_check.sort_values("timestamp").copy()

account_check["calculated_30d_avg"] = (
    account_check
    .set_index("timestamp")["amount"]
    .rolling("30D", min_periods=1)
    .mean()
    .values
)

display(
    account_check[
        [
            "transaction_id",
            "timestamp",
            "amount",
            "avg_transaction_amount_30d",
            "calculated_30d_avg",
            "amount_to_avg_ratio"
        ]
    ]
)

,transaction_id,timestamp,amount,avg_transaction_amount_30d,calculated_30d_avg,amount_to_avg_ratio
4131,FLR250227188863,2025-02-27 14:50:59,772325.00,-163006.50,7.723250e+05,4.74
4132,FLR250406126199,2025-04-06 15:45:23,5311245.00,-163006.50,5.311245e+06,32.58
4133,FLR250824103787,2025-08-24 14:00:47,15397962.62,-163006.50,1.539796e+07,94.46
4134,FLR250924141428,2025-09-24 22:54:16,-163006.50,-163006.50,-1.630065e+05,1.00
4135,FLR251022162041,2025-10-22 09:57:55,7335499.00,-163006.50,3.586246e+06,45.00
4136,FLR251209184177,2025-12-09 07:21:06,42607530.34,5113076.00,4.260753e+07,8.33
4137,FLR260322200710,2026-03-22 17:34:43,2065065.90,5113076.00,2.065066e+06,0.40
4138,FLR260623175283,2026-06-23 08:26:24,5113076.00,5113076.00,5.113076e+06,1.00
4139,FLR260623129250,2026-06-23 10:07:51,750619.59,5113076.00,2.931848e+06,0.15
4140,FLR260816215868,2026-08-16 08:51:28,174957.34,2475034.75,1.749573e+05,0.07


In [91]:
# Calculate the difference between the existing 30-day average
# and the independently calculated 30-day rolling average.

account_check["avg_difference"] = (
    account_check["avg_transaction_amount_30d"]
    - account_check["calculated_30d_avg"]
)

display(
    account_check[
        [
            "transaction_id",
            "timestamp",
            "avg_transaction_amount_30d",
            "calculated_30d_avg",
            "avg_difference"
        ]
    ]
)

,transaction_id,timestamp,avg_transaction_amount_30d,calculated_30d_avg,avg_difference
4131,FLR250227188863,2025-02-27 14:50:59,-163006.50,7.723250e+05,-9.353315e+05
4132,FLR250406126199,2025-04-06 15:45:23,-163006.50,5.311245e+06,-5.474252e+06
4133,FLR250824103787,2025-08-24 14:00:47,-163006.50,1.539796e+07,-1.556097e+07
4134,FLR250924141428,2025-09-24 22:54:16,-163006.50,-1.630065e+05,0.000000e+00
4135,FLR251022162041,2025-10-22 09:57:55,-163006.50,3.586246e+06,-3.749253e+06
4136,FLR251209184177,2025-12-09 07:21:06,5113076.00,4.260753e+07,-3.749445e+07
4137,FLR260322200710,2026-03-22 17:34:43,5113076.00,2.065066e+06,3.048010e+06
4138,FLR260623175283,2026-06-23 08:26:24,5113076.00,5.113076e+06,0.000000e+00
4139,FLR260623129250,2026-06-23 10:07:51,5113076.00,2.931848e+06,2.181228e+06
4140,FLR260816215868,2026-08-16 08:51:28,2475034.75,1.749573e+05,2.300077e+06


In [92]:
# Count how many records have a different calculated 30-day average
# from the existing avg_transaction_amount_30d feature.

different_avg = (
    account_check["avg_difference"].abs() > 0.01
).sum()

print(f"Records with different 30-day averages: {different_avg}")

Records with different 30-day averages: 8


In [93]:
# Calculate the 30-day historical average using only transactions
# that occurred before the current transaction.
# This avoids using the current transaction to calculate its own baseline.

account_check = account_check.sort_values("timestamp").copy()

account_check["previous_30d_avg"] = (
    account_check
    .set_index("timestamp")["amount"]
    .shift(1)
    .rolling("30D", min_periods=1)
    .mean()
    .values
)

display(
    account_check[
        [
            "transaction_id",
            "timestamp",
            "amount",
            "avg_transaction_amount_30d",
            "previous_30d_avg",
            "amount_to_avg_ratio"
        ]
    ]
)

,transaction_id,timestamp,amount,avg_transaction_amount_30d,previous_30d_avg,amount_to_avg_ratio
4131,FLR250227188863,2025-02-27 14:50:59,772325.00,-163006.50,NaN,4.74
4132,FLR250406126199,2025-04-06 15:45:23,5311245.00,-163006.50,772325.00,32.58
4133,FLR250824103787,2025-08-24 14:00:47,15397962.62,-163006.50,5311245.00,94.46
4134,FLR250924141428,2025-09-24 22:54:16,-163006.50,-163006.50,15397962.62,1.00
4135,FLR251022162041,2025-10-22 09:57:55,7335499.00,-163006.50,7617478.06,45.00
4136,FLR251209184177,2025-12-09 07:21:06,42607530.34,5113076.00,7335499.00,8.33
4137,FLR260322200710,2026-03-22 17:34:43,2065065.90,5113076.00,42607530.34,0.40
4138,FLR260623175283,2026-06-23 08:26:24,5113076.00,5113076.00,2065065.90,1.00
4139,FLR260623129250,2026-06-23 10:07:51,750619.59,5113076.00,3589070.95,0.15
4140,FLR260816215868,2026-08-16 08:51:28,174957.34,2475034.75,750619.59,0.07


In [94]:
# Calculate the difference between the dataset's existing average
# and the independently calculated previous-30-day average.

account_check["historical_avg_difference"] = (
    account_check["avg_transaction_amount_30d"]
    - account_check["previous_30d_avg"]
)

display(
    account_check[
        [
            "transaction_id",
            "avg_transaction_amount_30d",
            "previous_30d_avg",
            "historical_avg_difference"
        ]
    ]
)

,transaction_id,avg_transaction_amount_30d,previous_30d_avg,historical_avg_difference
4131,FLR250227188863,-163006.50,NaN,NaN
4132,FLR250406126199,-163006.50,772325.00,-935331.50
4133,FLR250824103787,-163006.50,5311245.00,-5474251.50
4134,FLR250924141428,-163006.50,15397962.62,-15560969.12
4135,FLR251022162041,-163006.50,7617478.06,-7780484.56
4136,FLR251209184177,5113076.00,7335499.00,-2222423.00
4137,FLR260322200710,5113076.00,42607530.34,-37494454.34
4138,FLR260623175283,5113076.00,2065065.90,3048010.10
4139,FLR260623129250,5113076.00,3589070.95,1524005.05
4140,FLR260816215868,2475034.75,750619.59,1724415.16


In [95]:
# Flag negative 30-day average values for further investigation.
# The original values are retained so that potentially useful information
# is not lost before the feature-generation logic is fully understood.

transactions["negative_avg_flag"] = (
    transactions["avg_transaction_amount_30d"] < 0
).astype(int)

print(
    "Transactions with negative 30-day averages:",
    transactions["negative_avg_flag"].sum()
)

Transactions with negative 30-day averages: 144


Note: Treatment of Negative 30-Day Average Values

Issue identified:
The avg_transaction_amount_30d column contained 144 negative values, which is unusual because a transaction average would normally be expected to be non-negative.

Investigation carried out:

Checked whether the negative values were associated with Reversed transactions — they were not; most were Completed.
Examined the affected accounts and found that some negative averages were repeated across multiple transactions.
Compared the stored avg_transaction_amount_30d with an independently calculated 30-day rolling average.
Also calculated a previous-30-day average excluding the current transaction.
The calculated values differed substantially from the stored values for many records, indicating that the issue could be related to the original feature-generation/calculation logic.

Resolution:
The negative values were not replaced with zero, absolute values, or NaN, because doing so could hide the underlying issue or introduce bias. Instead, the records were flagged for further investigation, while the original values were retained.

Easy explanation to remember:

“I detected → investigated → compared → found inconsistency → flagged rather than blindly changing the values.”

This method preserves the original data while making the data-quality issue explicit.

## Remove exact duplicate rows

Exact duplicate rows are removed. The number removed is recorded for transparency.

In [96]:
transactions_before = len(transactions)
accounts_before = len(accounts)

transactions = transactions.drop_duplicates().reset_index(drop=True)
accounts = accounts.drop_duplicates().reset_index(drop=True)

print(f"Transaction duplicates removed: {transactions_before - len(transactions):,}")
print(f"Account duplicates removed: {accounts_before - len(accounts):,}")

Transaction duplicates removed: 0
Account duplicates removed: 0


## Final validation

In [97]:
# Display a final summary of the transactions and accounts datasets after all cleaning steps
print("TRANSACTIONS AFTER CLEANING")
print(f"Rows: {len(transactions):,}")
print(f"Columns: {transactions.shape[1]:,}")
print(f"Remaining missing values: {transactions.isna().sum().sum():,}")
print(f"Remaining duplicate rows: {transactions.duplicated().sum():,}")

print("\nACCOUNTS AFTER CLEANING")
print(f"Rows: {len(accounts):,}")
print(f"Columns: {accounts.shape[1]:,}")
print(f"Remaining missing values: {accounts.isna().sum().sum():,}")
print(f"Remaining duplicate rows: {accounts.duplicated().sum():,}")

TRANSACTIONS AFTER CLEANING
Rows: 126,000
Columns: 25
Remaining missing values: 14,334
Remaining duplicate rows: 0

ACCOUNTS AFTER CLEANING
Rows: 7,200
Columns: 8
Remaining missing values: 0
Remaining duplicate rows: 0


In [98]:
# Check the number of missing values in each column of the transactions dataset
transactions.isnull().sum()

transaction_id                    0
account_id                        0
account_type                      0
kyc_tier                          0
timestamp                         0
day_of_week                       0
hour_of_day                       0
description                       0
merchant_name                     0
merchant_category                 0
channel                           0
amount                            0
currency                          0
amount_to_avg_ratio               0
avg_transaction_amount_30d        0
transaction_velocity_1h           0
transaction_country               0
home_country                      0
is_cross_border                   0
device_id                     14334
is_new_device                     0
account_age_days                  0
status                            0
is_fraud                          0
negative_avg_flag                 0
dtype: int64

In [99]:
# Replace missing device IDs with "Unknown" so that transactions are retained for analysis and fraud modelling.
transactions["device_id"] = transactions["device_id"].fillna("Unknown")

In [100]:
# Verify that all missing values have been successfully handled across the transactions dataset.
transactions.isnull().sum()

transaction_id                0
account_id                    0
account_type                  0
kyc_tier                      0
timestamp                     0
day_of_week                   0
hour_of_day                   0
description                   0
merchant_name                 0
merchant_category             0
channel                       0
amount                        0
currency                      0
amount_to_avg_ratio           0
avg_transaction_amount_30d    0
transaction_velocity_1h       0
transaction_country           0
home_country                  0
is_cross_border               0
device_id                     0
is_new_device                 0
account_age_days              0
status                        0
is_fraud                      0
negative_avg_flag             0
dtype: int64

In [101]:
# Display the final data types of all columns in the transactions and accounts datasets to confirm that the data is correctly formatted for analysis and modelling.

print("\nFinal transaction data types:")

print(transactions.dtypes)

print("\nFinal account data types:")

print(accounts.dtypes)


Final transaction data types:
transaction_id                        object
account_id                            object
account_type                          object
kyc_tier                              object
timestamp                     datetime64[ns]
day_of_week                           object
hour_of_day                            int64
description                           object
merchant_name                         object
merchant_category                     object
channel                               object
amount                               float64
currency                              object
amount_to_avg_ratio                  float64
avg_transaction_amount_30d           float64
transaction_velocity_1h                int64
transaction_country                   object
home_country                          object
is_cross_border                        int64
device_id                             object
is_new_device                        float64
account_age_days        

## 13. Save cleaned datasets

In [102]:
# Define the folder where the cleaned datasets will be stored
processed_folder = data_folder / "processed data"

# Create the folder if it does not already exist
processed_folder.mkdir(parents=True, exist_ok=True)

# Define the file paths for the cleaned transactions and accounts datasets
transactions_path = processed_folder / "finlora_transactions_clean.csv"
accounts_path = processed_folder / "finlora_accounts_clean.csv"

# Save the cleaned transactions and accounts datasets as CSV files in the processed data folder
transactions.to_csv(transactions_path, index=False)
accounts.to_csv(accounts_path, index=False)

# Display a confirmation message and the file paths where the cleaned datasets were saved
print("Cleaned datasets saved successfully:")

print(transactions_path)

print(accounts_path)

Cleaned datasets saved successfully:
C:\Users\telvi\Downloads\AMDARI\finlora_finance_flow\data\raw_data\processed data\finlora_transactions_clean.csv
C:\Users\telvi\Downloads\AMDARI\finlora_finance_flow\data\raw_data\processed data\finlora_accounts_clean.csv
